# 11 · Treino MLP (PyTorch puro) — Volume e Risco de OLA, D+1/D+7

Formaliza a abordagem validada nos testes exploratórios (`mlp_bobinho`
e as rodadas locais): **PyTorch puro, sem `TorchDistributor`** —
`toPandas()` traz o dado pro driver, o treino roda em Python/PyTorch
comum, sem tocar em nenhuma API de distribuição do Spark.

**Por que não distribuído:** confirmamos, em 4 tentativas diferentes
(`spark-excel`, `spark.conf.set` de Column Mapping, `SparkXGBRegressor`,
`TorchDistributor` com `local_mode=False`), que o compute serverless do
Databricks Free Edition bloqueia qualquer ferramenta que precise
introspeccionar configuração real do cluster
(`CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION` em todas). Não é falta de
sorte, é limitação de arquitetura do ambiente atual — documentada como
tal. Nossas tabelas são pequenas o bastante (a maior tem ~430 mil
linhas) pra caber inteira em memória via `toPandas()`, então essa
abordagem funciona mesmo sem cluster clássico.

**10 combinações treinadas** (6 de volume + 4 de risco de OLA), cada
uma com portão automático contra a baseline (`média móvel 7 dias`) —
só grava em Gold o que realmente ganhou.

**Arquitetura da rede**: `128 → 64 → 32 → 1`, com `Dropout(0.2)` nas
duas primeiras camadas, `Adam` (`lr=0.002`, `weight_decay=1e-4`), 250
épocas — validada empiricamente nos testes exploratórios (venceu a
baseline em 7 de 10 combinações testadas antes de formalizar aqui).

In [ ]:
%pip install -q torch
dbutils.library.restartPython()

In [ ]:
%run ./00_config

In [ ]:
from pyspark.sql import functions as F
from datetime import timedelta
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import mlflow

mlflow.set_experiment("/Shared/antecipeai_ml_experiments")

## Funções reutilizáveis

In [ ]:
DATA_INICIO_TREINO_ML = "2024-12-01"


def split_temporal_pandas(df, coluna_data="data_abertura", dias_teste=30, dias_validacao=30, dias_embargo=7):
    data_max = df[coluna_data].max()
    inicio_teste = data_max - timedelta(days=dias_teste - 1)
    fim_embargo_teste = inicio_teste - timedelta(days=dias_embargo)
    inicio_validacao = fim_embargo_teste - timedelta(days=dias_validacao - 1)
    fim_embargo_validacao = inicio_validacao - timedelta(days=dias_embargo)
    teste = df[df[coluna_data] >= inicio_teste].copy()
    validacao = df[(df[coluna_data] >= inicio_validacao) & (df[coluna_data] <= fim_embargo_teste)].copy()
    treino = df[df[coluna_data] <= fim_embargo_validacao].copy()
    return treino, validacao, teste


def construir_mlp(n_features):
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1),
    )


def treinar_e_avaliar(
    nome_tabela: str, target: str, coluna_categorica: str = None, epocas: int = 250, lr: float = 0.002
) -> dict:
    """
    Treina um MLP (PyTorch puro) para prever `target` a partir de uma tabela
    silver.features_series_*/features_risco_ola_*, com portão automático
    contra a baseline (média móvel 7d / taxa_media_movel_7d). Retorna tudo
    que a etapa de inferência (mais abaixo) precisa — inclusive a linha mais
    recente SEM dropna de alvo, pra não repetir o bug do "dia errado" que já
    corrigimos nos notebooks 08/09.
    """
    is_risco = "risco" in nome_tabela
    colunas_lag = (
        ["taxa_lag_1d", "taxa_lag_7d", "taxa_media_movel_7d", "taxa_media_movel_14d"]
        if is_risco else ["lag_1d", "lag_7d", "lag_14d", "media_movel_7d", "media_movel_14d"]
    )
    coluna_baseline = "taxa_media_movel_7d" if is_risco else "media_movel_7d"

    serie = spark.table(qualified_table(SCHEMA_SILVER, nome_tabela))
    calendario = spark.table(qualified_table(SCHEMA_SILVER, "features_calendario"))
    sdf = serie.join(calendario, "data_abertura", "left").filter(F.col("data_abertura") >= DATA_INICIO_TREINO_ML)
    sdf = sdf.withColumn("is_fim_de_semana_int", F.col("is_fim_de_semana").cast("int"))

    colunas_extra = ["qtd_kpi_regra_divergente", "qtd_duracao_suspeita"] if is_risco else ["qtd_kpi_regra_divergente"]
    colunas_features = colunas_lag + ["dia_semana_num", "trimestre", "is_feriado", "is_fim_de_semana_int"] + colunas_extra
    if coluna_categorica != "prioridade_num":
        colunas_features.append("prioridade_num")

    cols_select = ["data_abertura"] + colunas_features + [target]
    if coluna_baseline not in colunas_features:
        cols_select.append(coluna_baseline)
    if coluna_categorica:
        cols_select.append(coluna_categorica)

    pdf_bruto = sdf.select(*cols_select).toPandas()
    pdf_bruto["data_abertura"] = pdf_bruto["data_abertura"].astype("datetime64[ns]")

    if coluna_categorica:
        categorias = pdf_bruto[coluna_categorica].astype("category")
        pdf_bruto[f"{coluna_categorica}_idx"] = categorias.cat.codes
        colunas_features = colunas_features + [f"{coluna_categorica}_idx"]

    pdf = pdf_bruto.dropna(subset=colunas_lag + [target])

    treino, validacao, teste = split_temporal_pandas(pdf)
    treino_bruto, _, teste_bruto = split_temporal_pandas(pdf_bruto)

    medias = treino[colunas_features].mean()
    desvios = treino[colunas_features].std().replace(0, 1)

    def normalizar(df):
        X = ((df[colunas_features] - medias) / desvios).fillna(0).values.astype(np.float32)
        y = df[target].values.astype(np.float32).reshape(-1, 1)
        return torch.tensor(X), torch.tensor(y)

    X_treino, y_treino = normalizar(treino)
    X_val, y_val = normalizar(validacao)
    X_teste, y_teste = normalizar(teste)

    modelo = construir_mlp(len(colunas_features))
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr, weight_decay=1e-4)
    perda_fn = nn.MSELoss()

    for _ in range(epocas):
        modelo.train()
        otimizador.zero_grad()
        perda = perda_fn(modelo(X_treino), y_treino)
        perda.backward()
        otimizador.step()

    modelo.eval()
    with torch.no_grad():
        mae_val_modelo = (modelo(X_val) - y_val).abs().mean().item()
        mae_teste_modelo = (modelo(X_teste) - y_teste).abs().mean().item()

    mae_val_baseline = float(np.abs(validacao[coluna_baseline].values - validacao[target].values).mean())
    mae_teste_baseline = float(np.abs(teste[coluna_baseline].values - teste[target].values).mean())

    vencedor = "modelo" if mae_val_modelo < mae_val_baseline else "baseline"

    return {
        "tabela": nome_tabela, "target": target, "coluna_categorica": coluna_categorica,
        "vencedor": vencedor,
        "mae_val_modelo": round(mae_val_modelo, 4), "mae_val_baseline": round(mae_val_baseline, 4),
        "mae_teste_modelo": round(mae_teste_modelo, 4) if vencedor == "modelo" else None,
        "mae_teste_baseline": round(mae_teste_baseline, 4),
        "modelo": modelo if vencedor == "modelo" else None,
        "medias": medias, "desvios": desvios, "colunas_features": colunas_features,
        "coluna_baseline": coluna_baseline,
        "teste_bruto": teste_bruto,
        "is_risco": is_risco,
    }


print("Funções prontas.")

## Treinar as 10 combinações

In [ ]:
configuracoes_volume = [
    ("features_series_produto", "produto"),
    ("features_series_categoria", "categoria"),
    ("features_series_prioridade", None),
]
configuracoes_risco = [
    ("features_risco_ola_produto", "produto"),
    ("features_risco_ola_equipe", "grupo_designado"),
]

resultados = {}
for nome_tabela, cat in configuracoes_volume:
    for horizonte in ["target_d1", "target_d7"]:
        chave = f"{nome_tabela}__{horizonte}"
        print(f"Treinando: {chave}")
        with mlflow.start_run(run_name=chave):
            r = treinar_e_avaliar(nome_tabela, horizonte, cat)
            resultados[chave] = r
            mlflow.log_params({
                "tabela": nome_tabela, "target": horizonte, "coluna_categorica": cat,
                "arquitetura": "128-64-32", "dropout": 0.2, "lr": 0.002, "epocas": 250,
            })
            mlflow.log_metrics({
                "mae_val_modelo": r["mae_val_modelo"], "mae_val_baseline": r["mae_val_baseline"],
                "mae_teste_baseline": r["mae_teste_baseline"],
                **({"mae_teste_modelo": r["mae_teste_modelo"]} if r["mae_teste_modelo"] is not None else {}),
            })
            mlflow.set_tag("vencedor", r["vencedor"])
            mlflow.set_tag("tipo", "regressao_volume")

for nome_tabela, cat in configuracoes_risco:
    for horizonte in ["target_taxa_d1", "target_taxa_d7"]:
        chave = f"{nome_tabela}__{horizonte}"
        print(f"Treinando: {chave}")
        with mlflow.start_run(run_name=chave):
            r = treinar_e_avaliar(nome_tabela, horizonte, cat)
            resultados[chave] = r
            mlflow.log_params({
                "tabela": nome_tabela, "target": horizonte, "coluna_categorica": cat,
                "arquitetura": "128-64-32", "dropout": 0.2, "lr": 0.002, "epocas": 250,
            })
            mlflow.log_metrics({
                "mae_val_modelo": r["mae_val_modelo"], "mae_val_baseline": r["mae_val_baseline"],
                "mae_teste_baseline": r["mae_teste_baseline"],
                **({"mae_teste_modelo": r["mae_teste_modelo"]} if r["mae_teste_modelo"] is not None else {}),
            })
            mlflow.set_tag("vencedor", r["vencedor"])
            mlflow.set_tag("tipo", "regressao_risco_ola")

print(f"\n{len(resultados)} combinações treinadas. Experimentos em: /Shared/antecipeai_ml_experiments")

## Resumo comparativo

In [ ]:
resumo = pd.DataFrame([
    {"chave": k, "vencedor": v["vencedor"], "mae_val_modelo": v["mae_val_modelo"],
     "mae_val_baseline": v["mae_val_baseline"], "mae_teste_modelo": v["mae_teste_modelo"],
     "mae_teste_baseline": v["mae_teste_baseline"]}
    for k, v in resultados.items()
])
display(spark.createDataFrame(resumo))

qtd_modelo = sum(1 for v in resultados.values() if v["vencedor"] == "modelo")
print(f"\nModelo (MLP) venceu em {qtd_modelo} de {len(resultados)} combinações.")

## Inferência real (última data disponível) e gravação em Gold

Usa `teste_bruto` (sem `dropna` de alvo) pra pegar a linha do dia mais
recente de verdade — mesma correção já aplicada nos notebooks `08`/`09`.

In [ ]:
linhas_volume, linhas_risco = [], []

for chave, r in resultados.items():
    nome_tabela, horizonte = chave.split("__")
    teste_bruto = r["teste_bruto"]
    colunas_categoricas_reais = [c for c in [r["coluna_categorica"], "prioridade_num"] if c]

    ultima_por_segmento = (
        teste_bruto.sort_values("data_abertura")
        .groupby(colunas_categoricas_reais, as_index=False)
        .tail(1)
        .copy()
    )

    if r["vencedor"] == "modelo":
        X = ((ultima_por_segmento[r["colunas_features"]] - r["medias"]) / r["desvios"]).fillna(0).values.astype(np.float32)
        with torch.no_grad():
            valor_previsto = r["modelo"](torch.tensor(X)).numpy().flatten()
    else:
        valor_previsto = ultima_por_segmento[r["coluna_baseline"]].values

    dias_futuro = 1 if "d1" in horizonte else 7
    ultima_por_segmento["data_prevista"] = ultima_por_segmento["data_abertura"] + pd.Timedelta(days=dias_futuro)
    ultima_por_segmento["valor_previsto"] = np.round(valor_previsto, 4)
    ultima_por_segmento["horizonte"] = horizonte
    ultima_por_segmento["tabela_origem"] = nome_tabela
    ultima_por_segmento["metodo"] = r["vencedor"]

    if r["is_risco"]:
        linhas_risco.append(ultima_por_segmento)
    else:
        linhas_volume.append(ultima_por_segmento)

print(f"Combinações de volume: {len(linhas_volume)} | risco: {len(linhas_risco)}")

### Gravar `gold.previsoes_incidentes` (volume — sobrescreve com o resultado do MLP)

In [ ]:
def montar_saida_volume(df, coluna_categorica):
    saida = pd.DataFrame({
        "data_prevista": df["data_prevista"],
        "horizonte": df["horizonte"],
        "tabela_origem": df["tabela_origem"],
        "metodo": df["metodo"],
        "valor_previsto": df["valor_previsto"],
        "produto": df[coluna_categorica] if coluna_categorica == "produto" else None,
        "categoria": df[coluna_categorica] if coluna_categorica == "categoria" else None,
        "prioridade_num": df["prioridade_num"],
    })
    return saida


saidas_volume = []
for chave, r in resultados.items():
    if r["is_risco"]:
        continue
    idx = list(resultados.keys()).index(chave)
    df_pred = [ln for ln in linhas_volume if ln["tabela_origem"].iloc[0] == chave.split("__")[0] and ln["horizonte"].iloc[0] == chave.split("__")[1]][0]
    saidas_volume.append(montar_saida_volume(df_pred, r["coluna_categorica"]))

previsoes_volume_pdf = pd.concat(saidas_volume, ignore_index=True)
previsoes_volume_pdf["data_geracao"] = pd.Timestamp.now()

previsoes_volume_sdf = spark.createDataFrame(previsoes_volume_pdf)
(
    previsoes_volume_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_GOLD, "previsoes_incidentes"))
)
print(f"gold.previsoes_incidentes gravada: {previsoes_volume_sdf.count()} linhas")

### Gravar `gold.previsoes_risco_ola` (risco — tabela nova)

In [ ]:
def montar_saida_risco(df, coluna_categorica):
    saida = pd.DataFrame({
        "data_prevista": df["data_prevista"],
        "horizonte": df["horizonte"],
        "tabela_origem": df["tabela_origem"],
        "metodo": df["metodo"],
        "taxa_violacao_prevista": df["valor_previsto"],
        "produto": df[coluna_categorica] if coluna_categorica == "produto" else None,
        "grupo_designado": df[coluna_categorica] if coluna_categorica == "grupo_designado" else None,
        "prioridade_num": df["prioridade_num"],
    })
    return saida


saidas_risco = []
for chave, r in resultados.items():
    if not r["is_risco"]:
        continue
    df_pred = [ln for ln in linhas_risco if ln["tabela_origem"].iloc[0] == chave.split("__")[0] and ln["horizonte"].iloc[0] == chave.split("__")[1]][0]
    saidas_risco.append(montar_saida_risco(df_pred, r["coluna_categorica"]))

previsoes_risco_pdf = pd.concat(saidas_risco, ignore_index=True)
previsoes_risco_pdf["data_geracao"] = pd.Timestamp.now()

previsoes_risco_sdf = spark.createDataFrame(previsoes_risco_pdf)
(
    previsoes_risco_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_GOLD, "previsoes_risco_ola"))
)
print(f"gold.previsoes_risco_ola gravada: {previsoes_risco_sdf.count()} linhas")

In [ ]:
display(spark.table(qualified_table(SCHEMA_GOLD, "previsoes_incidentes")).orderBy("tabela_origem", "horizonte"))
display(spark.table(qualified_table(SCHEMA_GOLD, "previsoes_risco_ola")).orderBy("tabela_origem", "horizonte"))